# Team Do_Something | BUBT Bengali NLP Safety Challenge 2026
## Retraining on the FINAL RELEASE (public texts + private gold labels) and evaluating on the gold test labels

**What this notebook does (same pipeline and hyper-parameters as the competition notebook, only the data changes):**
1. Loads `train.csv`, `validation.csv`, `test.csv` and the private gold file (`test_gold_PRIVATE.csv`).
2. Pools train + validation (**3,792 texts**) and builds **5 leakage-free folds** (duplicate texts are kept in the same fold).
3. Trains BanglaBERT, XLM-R, BanglishBERT (5 folds each) and a TF-IDF logistic regression.
4. Tunes per-label blend weights and thresholds **on the out-of-fold (OOF) predictions only**.
5. Applies everything **once** to the test set and compares with the gold labels. **The gold labels are never used for training, checkpoint selection or tuning.**
6. Saves all numbers (`results.json`), predictions, and misclassified examples for the paper.

**How to attach the data on Kaggle:** Create a *private* Kaggle Dataset containing `public_dataset.zip` (Kaggle unzips it) **and** `test_gold_PRIVATE.csv`; then *Add Input* to this notebook. Turn on **GPU** (Settings > Accelerator). Do **not** attach the old competition data to the same notebook (file names clash).
Set `DEBUG_SMALL = True` first for a ~10-minute smoke test, then set it back to `False` for the real run.

## 1. Setup and data

In [1]:
import os, re, gc, glob, json, random, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, scipy.sparse as sp
from scipy.optimize import minimize
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

L = ['is_toxic','implicit_toxicity','sarcasm','contains_profanity','threatening','identity_attack','sexual_content']
SEED = 42
DEBUG_SMALL = False      # True = tiny/fast smoke test (600 train texts, 1 epoch). Set False for the real run.
DATA_DIR = None          # optional: e.g. "/kaggle/input/my-dataset"; None = auto-search
OUT = "/kaggle/working" if os.path.isdir("/kaggle/working") else "outputs"
os.makedirs(OUT, exist_ok=True)
random.seed(SEED); np.random.seed(SEED)

def clean(t):
    return re.sub(r"\s+", " ", str(t)).strip()

import zipfile
def _roots():
    return ([DATA_DIR] if DATA_DIR else ["/kaggle/input", "/content", "."]) + [f"{OUT}/_unzipped"]

def show_tree(root="/kaggle/input", depth=3):
    print(f"--- what is inside {root} (depth {depth}) ---")
    if not os.path.isdir(root): print("  (folder does not exist)"); return
    base = root.rstrip("/").count("/")
    for r, ds, fs in os.walk(root):
        if r.count("/") - base >= depth: ds[:] = []
        print("  ", r, "|", fs[:8])

def unpack_zips():
    for r in _roots():
        if not os.path.isdir(r): continue
        for z in glob.glob(f"{r}/**/*.zip", recursive=True):
            dest = f"{OUT}/_unzipped/{os.path.splitext(os.path.basename(z))[0]}"
            if not os.path.isdir(dest):
                zipfile.ZipFile(z).extractall(dest); print("unzipped", z, "->", dest)

def find(pattern):
    def search():
        hits = {}
        for r in _roots():
            if os.path.isdir(r):
                for p in glob.glob(f"{r}/**/{pattern}", recursive=True): hits[os.path.realpath(p)] = p
        return sorted(hits.values())
    hits = search()
    if not hits: unpack_zips(); hits = search()          # dataset uploaded as a .zip that Kaggle did not extract
    if not hits:
        show_tree("/kaggle/input"); show_tree(".", 2)
        raise FileNotFoundError(f"'{pattern}' not found. In the right-hand panel: Add Input -> your private dataset, "
                                f"then re-run. (Or set DATA_DIR to the folder shown above.)")
    if len(hits) > 1: print(f"WARNING: several matches for {pattern}: {hits} -> using {hits[0]}")
    print(f"{pattern:15s} -> {hits[0]}")
    return hits[0]

tr  = pd.read_csv(find("train.csv"))
va  = pd.read_csv(find("validation.csv"))
te_pub = pd.read_csv(find("test.csv"))
gold   = pd.read_csv(find("*gold*.csv"))

assert len(te_pub) == len(gold) and set(te_pub.ID) == set(gold.ID), "gold IDs do not match test.csv IDs"
gold = gold.set_index("ID").loc[te_pub.ID].reset_index()
Y_test = gold[L].values.astype(int)            # PRIVATE labels: used ONLY for the final evaluation
te = te_pub[["ID", "text"]].copy()             # test texts without labels
al = pd.concat([tr, va], ignore_index=True)    # training pool (train + validation)

if DEBUG_SMALL:
    al = al.sample(600, random_state=SEED).reset_index(drop=True)
    te = te.iloc[:200].reset_index(drop=True); Y_test = Y_test[:200]

for d in (al, te): d["text"] = d["text"].fillna("").apply(clean)
Y = al[L].values.astype(int)
print("pool:", al.shape, "| test:", te.shape, "| DEBUG_SMALL =", DEBUG_SMALL)

train.csv       -> /kaggle/input/datasets/serajummunira29/private-dataset-final/train.csv
validation.csv  -> /kaggle/input/datasets/serajummunira29/private-dataset-final/validation.csv
test.csv        -> /kaggle/input/datasets/serajummunira29/private-dataset-final/test.csv
*gold*.csv      -> /kaggle/input/datasets/serajummunira29/private-dataset-final/test_gold_PRIVATE.csv
pool: (3792, 9) | test: (1000, 2) | DEBUG_SMALL = False


## 2. Dataset statistics for the paper (whole release: public + private)

In [2]:
STATS = {}
def counts(d): return d[L].sum().astype(int).tolist()
rel = pd.concat([tr.assign(split="train"), va.assign(split="val"),
                 te_pub.merge(gold, on="ID").assign(split="test")], ignore_index=True)
rel["text"] = rel.text.fillna("").apply(clean)

tab = pd.DataFrame({s: counts(rel[rel.split == s]) for s in ["train", "val", "test"]}, index=L)
tab["total"] = tab.sum(1); tab["rate_%"] = (tab.total / len(rel) * 100).round(1)
print("=== Table: positives per label and split (final release) ===")
print({s: int((rel.split == s).sum()) for s in ["train", "val", "test"]}, "total", len(rel))
print(tab, "\n")
STATS["sizes"] = {s: int((rel.split == s).sum()) for s in ["train", "val", "test"]}
STATS["label_table"] = tab.reset_index().rename(columns={"index": "label"}).to_dict("list")

pool = rel[rel.split != "test"]
print("P(is_toxic | label) in release:", {l: round(float(rel.loc[rel[l] == 1, "is_toxic"].mean()), 3) for l in L[1:]})
print("no label: %.3f | more than one label: %.3f" % ((rel[L].sum(1) == 0).mean(), (rel[L].sum(1) > 1).mean()))

t = rel.text; sz = t.value_counts()
same = rel.groupby("text")[L].apply(lambda d: len(d.drop_duplicates()) == 1)
print("unique texts (all): %d of %d | duplicate groups: %d | rows in duplicate groups: %d" % (t.nunique(), len(rel), (sz > 1).sum(), t.duplicated(keep=False).sum()))
print("duplicate groups with identical labels: %.3f" % same[sz[sz > 1].index].mean())
trt, vat, tet = (set(rel[rel.split == s].text) for s in ["train", "val", "test"])
print("train-val shared texts:", len(trt & vat), "| test texts also in train+val:", len(tet & (trt | vat)))
maj = pool.groupby("text")[L].agg(lambda s: int(s.mean() > 0.5))
tt = rel[(rel.split == "test") & rel.text.isin(maj.index)]
agree = (tt[L].values == maj.loc[tt.text].values).all(1).mean() if len(tt) else float("nan")
print("test texts seen in pool: %d | full label vector equals pool majority: %.3f" % (len(tt), agree))

def latin_frac(s):
    b = len(re.findall(r"[\u0980-\u09FF]", s)); l = len(re.findall(r"[A-Za-z]", s))
    return l / (b + l) if b + l else 0.0
lf = rel.text.apply(latin_frac)
print("script: Bengali %.3f | mixed %.3f | romanised %.3f" % ((lf < .05).mean(), ((lf >= .05) & (lf <= .95)).mean(), (lf > .95).mean()))
print("chars: median %.0f, p95 %.0f | words mean %.1f" % (t.str.len().median(), t.str.len().quantile(.95), t.str.split().str.len().mean()))
STATS["other"] = dict(unique=int(t.nunique()), dup_groups=int((sz > 1).sum()), dup_rows=int(t.duplicated(keep=False).sum()),
                      dup_identical=float(same[sz[sz > 1].index].mean()), train_val_overlap=len(trt & vat),
                      test_in_pool=len(tet & (trt | vat)), test_pool_label_agree=float(agree),
                      bengali=float((lf < .05).mean()), mixed=float(((lf >= .05) & (lf <= .95)).mean()), romanised=float((lf > .95).mean()))

=== Table: positives per label and split (final release) ===
{'train': 3292, 'val': 500, 'test': 1000} total 4792
                    train  val  test  total  rate_%
is_toxic             2203  335   669   3207    66.9
implicit_toxicity     194   30    59    283     5.9
sarcasm               213   32    64    309     6.4
contains_profanity    461   70   140    671    14.0
threatening           135   20    41    196     4.1
identity_attack       424   64   129    617    12.9
sexual_content        113   17    34    164     3.4 

P(is_toxic | label) in release: {'implicit_toxicity': 0.993, 'sarcasm': 0.955, 'contains_profanity': 1.0, 'threatening': 1.0, 'identity_attack': 1.0, 'sexual_content': 1.0}
no label: 0.327 | more than one label: 0.335
unique texts (all): 3811 of 4792 | duplicate groups: 902 | rows in duplicate groups: 1883
duplicate groups with identical labels: 0.529
train-val shared texts: 141 | test texts also in train+val: 312
test texts seen in pool: 317 | full label vector e

## 3. Leakage-free folds (identical texts never cross folds)

In [3]:
groups = pd.factorize(al.text)[0]
ystrat = (Y[:, 1:].sum(1) > 0).astype(int) * 2 + Y[:, 0]
FOLDS = list(StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED).split(al, ystrat, groups))
for k, (a, b) in enumerate(FOLDS):
    assert not (set(groups[a]) & set(groups[b])), "duplicate text across folds!"
    print(f"fold {k+1}: fit {len(a)} | held-out {len(b)} | positives in held-out {Y[b].sum(0).tolist()}")

fold 1: fit 3039 | held-out 753 | positives in held-out [514, 41, 46, 122, 26, 101, 34]
fold 2: fit 3040 | held-out 752 | positives in held-out [500, 29, 43, 109, 27, 95, 21]
fold 3: fit 3025 | held-out 767 | positives in held-out [502, 57, 50, 97, 31, 108, 25]
fold 4: fit 3028 | held-out 764 | positives in held-out [509, 58, 61, 97, 37, 89, 23]
fold 5: fit 3036 | held-out 756 | positives in held-out [513, 39, 45, 106, 34, 95, 27]


## 4. Transformer code (unchanged from the competition notebook)

In [4]:
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
EPOCHS = 1 if DEBUG_SMALL else 7
print("device:", DEVICE, "| epochs:", EPOCHS)

class ToxDS(Dataset):
    def __init__(s, df, tok, max_len, lab=True):
        s.enc = tok(df.text.tolist(), truncation=True, max_length=max_len)
        s.y = df[L].values.astype("float32") if lab else None
    def __len__(s): return len(s.enc["input_ids"])
    def __getitem__(s, i):
        d = {k: v[i] for k, v in s.enc.items()}
        if s.y is not None: d["labels"] = s.y[i]
        return d

def make_collate(tok):
    def collate(b):
        m = max(len(x["input_ids"]) for x in b)
        out = {"input_ids": torch.tensor([x["input_ids"] + [tok.pad_token_id]*(m-len(x["input_ids"])) for x in b]),
               "attention_mask": torch.tensor([[1]*len(x["input_ids"]) + [0]*(m-len(x["input_ids"])) for x in b])}
        if "labels" in b[0]: out["labels"] = torch.tensor(np.stack([x["labels"] for x in b]))
        return out
    return collate

class ToxModel(nn.Module):
    def __init__(s, model_name):
        super().__init__()
        s.enc = AutoModel.from_pretrained(model_name)
        h = s.enc.config.hidden_size
        s.drops = nn.ModuleList([nn.Dropout(p) for p in [0.1, 0.2, 0.3, 0.4, 0.5]])
        s.head = nn.Linear(h * 2, 7)
    def forward(s, input_ids, attention_mask):
        out = s.enc(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        m = attention_mask.unsqueeze(-1).float()
        mean_pooled = (out * m).sum(1) / m.sum(1).clamp(min=1e-6)
        max_pooled = torch.max(out.masked_fill(m == 0, -1e9), dim=1)[0]
        pooled = torch.cat([mean_pooled, max_pooled], dim=1)
        return torch.stack([s.head(d(pooled)) for d in s.drops], dim=0).mean(0)

def f1_total(Yt, P, thr=0.5):
    f = [f1_score(Yt[:, j], P[:, j] >= thr, average="macro" if j == 0 else "binary", zero_division=0) for j in range(7)]
    return 0.5 * f[0] + 0.5 * np.mean(f[1:]), f

@torch.no_grad()
def predict(model, ds, collate, bs=64):
    model.eval(); P = []
    for b in DataLoader(ds, batch_size=bs, collate_fn=collate):
        b = {k: v.to(DEVICE) for k, v in b.items()}
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=DEVICE.type == "cuda"):
            p = torch.sigmoid(model(b["input_ids"], b["attention_mask"]))
        P.append(p.float().cpu().numpy())
    return np.concatenate(P)

def train_fold(model_name, tr_df, va_df, epochs=7, bs=16, lr=2e-5, max_len=160):
    tok = AutoTokenizer.from_pretrained(model_name); collate = make_collate(tok)
    dtr, dva = ToxDS(tr_df, tok, max_len), ToxDS(va_df, tok, max_len)
    model = ToxModel(model_name).to(DEVICE)
    pos = tr_df[L].values.mean(0)
    pos_w = torch.tensor(np.clip((1 - pos) / np.clip(pos, 1e-6, None), 1.0, 10.0), dtype=torch.float32).to(DEVICE)
    lossf = nn.BCEWithLogitsLoss(pos_weight=pos_w)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    loader = DataLoader(dtr, batch_size=bs, shuffle=True, collate_fn=collate)
    total = len(loader) * epochs
    sched = get_cosine_schedule_with_warmup(opt, int(0.1 * total), total)
    scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda")
    best_score, best_probs, best_model = -1, None, None
    for ep in range(epochs):
        model.train()
        for b in loader:
            b = {k: v.to(DEVICE) for k, v in b.items()}
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=DEVICE.type == "cuda"):
                loss = lossf(model(b["input_ids"], b["attention_mask"]).float(), b["labels"])
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
        pv = predict(model, dva, collate)
        s, _ = f1_total(va_df[L].values, pv)
        print(f"  epoch {ep+1}/{epochs} score@0.5={s:.4f}", flush=True)
        if s > best_score: best_score, best_probs, best_model = s, pv, {k: v.cpu() for k, v in model.state_dict().items()}
    model.load_state_dict(best_model)
    return model, tok, collate, best_probs, best_score

def run_cv(model_name, tag, epochs=None, bs=16, lr=2e-5, max_len=160):
    epochs = epochs or EPOCHS
    done = f"{OUT}/cv_{tag}.npz"
    if os.path.exists(done):                       # resume support: skip finished models
        z = np.load(done); print(f"{tag}: loaded finished run from {done}"); return z["oof"], z["test"]
    oof = np.zeros((len(al), 7), dtype="float32"); test_accum = np.zeros((len(te), 7), dtype="float32")
    best_scores = []
    for fold, (a, b) in enumerate(FOLDS):
        print(f"--- {tag} fold {fold+1}/{len(FOLDS)} ---")
        model, tok, collate, val_probs, val_score = train_fold(model_name, al.iloc[a], al.iloc[b], epochs, bs, lr, max_len)
        oof[b] = val_probs; best_scores.append(val_score)
        test_accum += predict(model, ToxDS(te, tok, max_len, lab=False), collate) / len(FOLDS)
        del model; gc.collect(); torch.cuda.empty_cache()
    print(f"{tag}: per-fold best scores {np.round(best_scores, 4).tolist()}")
    np.savez(done, oof=oof, test=test_accum)
    return oof, test_accum

device: cuda | epochs: 7


## 5. Train the three transformers (5 folds each, GPU)

In [14]:
oof_bb, test_bb = run_cv("csebuetnlp/banglabert", "banglabert")

banglabert: loaded finished run from /kaggle/working/cv_banglabert.npz


In [10]:
oof_xlmr, test_xlmr = run_cv("xlm-roberta-base", "xlmr")

--- xlmr fold 1/5 ---


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  epoch 1/7 score@0.5=0.3871
  epoch 2/7 score@0.5=0.4214
  epoch 3/7 score@0.5=0.4843
  epoch 4/7 score@0.5=0.4777
  epoch 5/7 score@0.5=0.4830
  epoch 6/7 score@0.5=0.4942
  epoch 7/7 score@0.5=0.4955
--- xlmr fold 2/5 ---


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  epoch 1/7 score@0.5=0.2768
  epoch 2/7 score@0.5=0.4152
  epoch 3/7 score@0.5=0.4774
  epoch 4/7 score@0.5=0.5001
  epoch 5/7 score@0.5=0.5046
  epoch 6/7 score@0.5=0.5043
  epoch 7/7 score@0.5=0.5052
--- xlmr fold 5/5 ---


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  epoch 1/7 score@0.5=0.2442
  epoch 2/7 score@0.5=0.4616
  epoch 3/7 score@0.5=0.4764
  epoch 4/7 score@0.5=0.5106
  epoch 5/7 score@0.5=0.5053
  epoch 6/7 score@0.5=0.5035
  epoch 7/7 score@0.5=0.5124
xlmr: per-fold best scores [0.4955, 0.4858, 0.4879, 0.5052, 0.5124]


In [15]:
oof_blish, test_blish = run_cv("csebuetnlp/banglishbert", "banglishbert")

--- banglishbert fold 1/5 ---


config.json:   0%|          | 0.00/874 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/366k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

[transformers] ElectraModel LOAD REPORT from: csebuetnlp/banglishbert
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
discriminator_predictions.dense.weight            | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED |  | 
discriminator_predictions.dense.bias              | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  epoch 1/7 score@0.5=0.2604
  epoch 2/7 score@0.5=0.4041
  epoch 3/7 score@0.5=0.4452
  epoch 4/7 score@0.5=0.4834
  epoch 5/7 score@0.5=0.5111
  epoch 6/7 score@0.5=0.5083
  epoch 7/7 score@0.5=0.5115
--- banglishbert fold 5/5 ---


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] ElectraModel LOAD REPORT from: csebuetnlp/banglishbert
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
discriminator_predictions.dense.weight            | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED |  | 
discriminator_predictions.dense.bias              | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  epoch 1/7 score@0.5=0.2685
  epoch 2/7 score@0.5=0.3695
  epoch 3/7 score@0.5=0.4784
  epoch 4/7 score@0.5=0.5048
  epoch 5/7 score@0.5=0.4972
  epoch 6/7 score@0.5=0.5045
  epoch 7/7 score@0.5=0.5072
banglishbert: per-fold best scores [0.5028, 0.4994, 0.5079, 0.5115, 0.5072]


## 6. TF-IDF + logistic regression branch (same folds)

In [16]:
def tfidf_cv():
    oof = np.zeros((len(al), 7)); test_accum = np.zeros((len(te), 7))
    for a, b in FOLDS:
        w = TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=2, sublinear_tf=True, token_pattern=r"(?u)\b\w+\b")
        c = TfidfVectorizer(analyzer="char_wb", ngram_range=(1, 5), min_df=2, sublinear_tf=True, max_features=300000)
        Xa = sp.hstack([w.fit_transform(al.text.iloc[a]), c.fit_transform(al.text.iloc[a])]).tocsr()
        Xb = sp.hstack([w.transform(al.text.iloc[b]), c.transform(al.text.iloc[b])]).tocsr()
        Xt = sp.hstack([w.transform(te.text), c.transform(te.text)]).tocsr()
        for j in range(7):
            m = LogisticRegression(C=1, class_weight="balanced", max_iter=2000).fit(Xa, Y[a, j])
            oof[b, j] = m.predict_proba(Xb)[:, 1]
            test_accum[:, j] += m.predict_proba(Xt)[:, 1] / len(FOLDS)
    return oof, test_accum

oof_tf, test_tf = tfidf_cv()
np.savez(f"{OUT}/cv_tfidf.npz", oof=oof_tf, test=test_tf)
print("TF-IDF OOF done")

TF-IDF OOF done


## 7. Per-label blend weights and thresholds (tuned on OOF only)

In [17]:
oofs  = {"banglabert": oof_bb, "xlmr": oof_xlmr, "banglishbert": oof_blish, "tfidf": oof_tf}
tests = {"banglabert": test_bb, "xlmr": test_xlmr, "banglishbert": test_blish, "tfidf": test_tf}
names = list(oofs.keys()); M = len(names)

best_weights = np.zeros((7, M)); best_thresholds = np.zeros(7)
oof_blend = np.zeros(Y.shape, dtype=np.float32); test_blend = np.zeros((len(te), 7), dtype=np.float32)
for j in range(7):
    avg = "macro" if j == 0 else "binary"
    pj  = np.column_stack([oofs[n][:, j] for n in names])
    ptj = np.column_stack([tests[n][:, j] for n in names])
    def objective(p):
        w = p[:M]; w = w / w.sum() if w.sum() > 0 else np.ones(M) / M
        return -f1_score(Y[:, j], (pj @ w >= p[M]).astype(int), average=avg, zero_division=0)
    res = minimize(objective, np.array([1.0 / M] * M + [0.40]), method="Powell", bounds=[(0, 1)] * M + [(0.05, 0.85)])
    w = res.x[:M]; w = w / w.sum() if w.sum() > 0 else np.ones(M) / M
    best_weights[j] = w; best_thresholds[j] = np.clip(res.x[M], 0.05, 0.85)
    oof_blend[:, j] = pj @ w; test_blend[:, j] = ptj @ w

print("Thresholds:", dict(zip(L, np.round(best_thresholds, 3).tolist())))
print(pd.DataFrame(best_weights, index=L, columns=names).round(3))

# simple "safe" 50/50 BanglaBERT + TF-IDF blend with a smoothed threshold search
Pv_safe = 0.5 * oofs["banglabert"] + 0.5 * oofs["tfidf"]; Pt_safe = 0.5 * tests["banglabert"] + 0.5 * tests["tfidf"]
ts = np.arange(0.05, 0.95, 0.01); T_safe = []
for j in range(7):
    f = np.array([f1_score(Y[:, j], Pv_safe[:, j] >= t, average="macro" if j == 0 else "binary", zero_division=0) for t in ts])
    T_safe.append(ts[int(np.argmax(np.convolve(f, np.ones(7) / 7, mode="same")))])
T_safe = np.array(T_safe)

Thresholds: {'is_toxic': 0.53, 'implicit_toxicity': 0.409, 'sarcasm': 0.46, 'contains_profanity': 0.502, 'threatening': 0.536, 'identity_attack': 0.514, 'sexual_content': 0.515}
                    banglabert   xlmr  banglishbert  tfidf
is_toxic                 0.464  0.222         0.042  0.272
implicit_toxicity        0.295  0.334         0.297  0.074
sarcasm                  0.315  0.277         0.093  0.315
contains_profanity       0.308  0.060         0.085  0.547
threatening              0.626  0.091         0.166  0.117
identity_attack          0.131  0.143         0.151  0.576
sexual_content           0.122  0.164         0.183  0.531


## 8. Evaluation: OOF (pool) and TEST (gold labels)
Two summary scores are printed for every system:
* **S** = internal composite (0.5 x macro-F1 of `is_toxic` + 0.5 x mean binary F1 of the other six) - the score we used for model selection.
* **M** = mean over the seven labels of the macro-F1 (both classes) - our best guess of the organisers' leaderboard metric. **Confirm the exact official formula with the organisers** and add it below if it differs.

In [18]:
def scores(Yt, P):
    macro = [f1_score(Yt[:, j], P[:, j], average="macro", zero_division=0) for j in range(7)]
    binf  = [f1_score(Yt[:, j], P[:, j], zero_division=0) for j in range(7)]
    prec  = [precision_score(Yt[:, j], P[:, j], zero_division=0) for j in range(7)]
    rec   = [recall_score(Yt[:, j], P[:, j], zero_division=0) for j in range(7)]
    S = 0.5 * macro[0] + 0.5 * np.mean(binf[1:]); Mm = float(np.mean(macro))
    return dict(S=float(S), M=Mm, macro=macro, binary=binf, precision=prec, recall=rec)

systems = {}
for n in ["tfidf", "xlmr", "banglishbert", "banglabert"]:
    systems[n] = ((oofs[n] >= 0.5).astype(int), (tests[n] >= 0.5).astype(int))
systems["safe_blend_50_50"] = ((Pv_safe >= T_safe).astype(int), (Pt_safe >= T_safe).astype(int))
systems["final_blend"] = ((oof_blend >= best_thresholds).astype(int), (test_blend >= best_thresholds).astype(int))

RES = {"systems": {}}
rows = []
for n, (po, pt) in systems.items():
    so, st = scores(Y, po), scores(Y_test, pt)
    RES["systems"][n] = {"oof": so, "test": st}
    rows.append([n, so["S"], so["M"], st["S"], st["M"]])
summary = pd.DataFrame(rows, columns=["system", "OOF_S", "OOF_M", "TEST_S", "TEST_M"]).round(4)
print("=== System comparison (OOF = 5-fold pool, TEST = gold labels) ===")
print(summary.to_string(index=False))

# per-label table for the final blend on TEST, with the trivial all-positive baseline
pt = systems["final_blend"][1]; st = RES["systems"]["final_blend"]["test"]
pl = pd.DataFrame({"label": L, "true_rate": Y_test.mean(0), "pred_rate": pt.mean(0),
                   "precision": st["precision"], "recall": st["recall"], "F1_binary": st["binary"], "F1_macro": st["macro"],
                   "trivial_all_pos_F1": [2 * p / (1 + p) for p in Y_test.mean(0)], "threshold": best_thresholds}).round(4)
print("\n=== Final blend, per label, TEST ===")
print(pl.to_string(index=False))
so = RES["systems"]["final_blend"]["oof"]
print("\n=== Final blend, per label, OOF (F1 binary; is_toxic = macro) ===")
print(pd.DataFrame({"label": L, "F1": [so["macro"][0]] + so["binary"][1:], "banglabert_F1@0.5": [RES["systems"]["banglabert"]["oof"]["macro"][0]] + RES["systems"]["banglabert"]["oof"]["binary"][1:],
                    "oof_pred_rate": systems["final_blend"][0].mean(0), "oof_true_rate": Y.mean(0)}).round(4).to_string(index=False))
RES["per_label_test_final"] = pl.to_dict("list"); RES["thresholds"] = dict(zip(L, best_thresholds.tolist()))
RES["weights"] = pd.DataFrame(best_weights, index=L, columns=names).to_dict()

=== System comparison (OOF = 5-fold pool, TEST = gold labels) ===
          system  OOF_S  OOF_M  TEST_S  TEST_M
           tfidf 0.5135 0.6520  0.5492  0.6796
            xlmr 0.4975 0.6244  0.5257  0.6560
    banglishbert 0.5073 0.6480  0.5509  0.6828
      banglabert 0.5392 0.6696  0.5660  0.6823
safe_blend_50_50 0.5626 0.6844  0.5877  0.7044
     final_blend 0.5764 0.6978  0.5894  0.7065

=== Final blend, per label, TEST ===
             label  true_rate  pred_rate  precision  recall  F1_binary  F1_macro  trivial_all_pos_F1  threshold
          is_toxic      0.669      0.668     0.8159  0.8146     0.8153    0.7214              0.8017     0.5296
 implicit_toxicity      0.059      0.120     0.1417  0.2881     0.1899    0.5552              0.1114     0.4090
           sarcasm      0.064      0.103     0.2427  0.3906     0.2994    0.6178              0.1203     0.4605
contains_profanity      0.140      0.172     0.6802  0.8357     0.7500    0.8519              0.2456     0.5019
       

## 9. Extra analyses for the paper (test set)

In [19]:
pt = systems["final_blend"][1]
seen = te.text.isin(set(al.text)).values
print("Test texts that also occur in the training pool:", int(seen.sum()), "of", len(te))
extra = {}
for name, mask in [("seen_in_pool", seen), ("unseen", ~seen)]:
    if mask.sum() > 10:
        r = scores(Y_test[mask], pt[mask]); extra[name] = dict(n=int(mask.sum()), S=r["S"], M=r["M"])
        print(f"  {name:13s} n={mask.sum():4d}  S={r['S']:.4f}  M={r['M']:.4f}")
lf_te = te.text.apply(latin_frac)
for name, mask in [("Bengali script", (lf_te < .05).values), ("mixed / Latin", (lf_te >= .05).values)]:
    if mask.sum() > 10:
        r = scores(Y_test[mask], pt[mask]); extra[name] = dict(n=int(mask.sum()), S=r["S"], M=r["M"], f1_is_toxic=r["macro"][0], f1_profanity=r["binary"][3])
        print(f"  {name:14s} n={mask.sum():4d}  S={r['S']:.4f}  M={r['M']:.4f}  is_toxic macro-F1={r['macro'][0]:.4f}  profanity F1={r['binary'][3]:.4f}")
RES["extra"] = extra; RES["stats"] = STATS

Test texts that also occur in the training pool: 317 of 1000
  seen_in_pool  n= 317  S=0.5951  M=0.7231
  unseen        n= 683  S=0.5845  M=0.7000
  Bengali script n= 720  S=0.5938  M=0.7016  is_toxic macro-F1=0.7463  profanity F1=0.7677
  mixed / Latin  n= 280  S=0.5572  M=0.7068  is_toxic macro-F1=0.6357  profanity F1=0.7193


## 10. Misclassified test examples (for the Error Analysis section)

In [20]:
rows = []
for j, lab in enumerate(L):
    p = test_blend[:, j]; y = Y_test[:, j]; yh = pt[:, j]
    fp = np.where((yh == 1) & (y == 0))[0]; fn = np.where((yh == 0) & (y == 1))[0]
    for i in fp[np.argsort(-p[fp])][:4]: rows.append((lab, "false_positive", te.ID[i], round(float(p[i]), 3), te.text[i], Y_test[i].tolist()))
    for i in fn[np.argsort(p[fn])][:4]:  rows.append((lab, "false_negative", te.ID[i], round(float(p[i]), 3), te.text[i], Y_test[i].tolist()))
err = pd.DataFrame(rows, columns=["label", "type", "ID", "blend_prob", "text", "true_labels(7)"])
err.to_csv(f"{OUT}/error_examples.csv", index=False, encoding="utf-8-sig")
pd.set_option("display.max_colwidth", 110); pd.set_option("display.width", 220)
show = err.drop(columns=["true_labels(7)"]).copy(); show["text"] = show.text.str.slice(0, 100)
print(show.to_string(index=False))

             label           type        ID  blend_prob                                                                                                 text
          is_toxic false_positive BICH_3409       0.881                                            Tui to khanki giri korish tui ki samajer valo kichu bujis
          is_toxic false_positive BICH_2971       0.878                                                            Ja khanki ir chele age bra pore video kor
          is_toxic false_positive BICH_3652       0.866                                                   Tui akta madar chod ar toa rachian vabi randi humo
          is_toxic false_positive BICH_3673       0.854                                        একটা চোদার থেকে ১০০ টার চোদা বেশি হই যায় মনে হয় নাটক চোদায়
          is_toxic false_negative BICH_1861       0.204 উনি ঠিক করেছে সবাই আমরা যারা ভালো আছি যারা এসব নোংরামো করি না তারা সবাই উনার পাশে আছে আর তোমার মত গি
          is_toxic false_negative BICH_1514       0.219   

## 11. Save everything (upload `results.json` and `error_examples.csv` to the paper assistant)

In [21]:
sub = pd.DataFrame(pt, columns=L); sub.insert(0, "ID", te.ID.values)
sub.to_csv(f"{OUT}/test_predictions_final_release.csv", index=False)
RES["config"] = dict(seed=SEED, folds="StratifiedGroupKFold(5, shuffle, group=clean text)", pool=len(al), test=len(te),
                     epochs=EPOCHS if "EPOCHS" in globals() else None, batch=16, lr=2e-5, max_len=160, debug_small=DEBUG_SMALL)
def tojson(o):
    if isinstance(o, (np.floating, np.integer)): return o.item()
    if isinstance(o, np.ndarray): return o.tolist()
    raise TypeError(type(o))
with open(f"{OUT}/results.json", "w", encoding="utf-8") as f: json.dump(RES, f, ensure_ascii=False, indent=1, default=tojson)
print("Saved:", sorted(os.listdir(OUT)))
print("\n>>> Final blend on gold test labels:  S = %.4f   M = %.4f" % (RES["systems"]["final_blend"]["test"]["S"], RES["systems"]["final_blend"]["test"]["M"]))

Saved: ['.virtual_documents', 'cv_banglabert.npz', 'cv_banglishbert.npz', 'cv_tfidf.npz', 'cv_xlmr.npz', 'error_examples.csv', 'results.json', 'test_predictions_final_release.csv']

>>> Final blend on gold test labels:  S = 0.5894   M = 0.7065
